Trick: read `const ...` left to right

```cpp
vector<int> vec{1, 2, 3};

const vector<int> const_vec{1, 2, 3};
const_vec.push_back(3);                 // NOT OK

vector<int>& ref_vec{vec};
ref_vec.push_back(3);                   // OK

const vector<int>& const_ref{vec};
const_ref.push_back(3);                 // NOT OK

vector<int>& test_vec{const_vec};       // NOT OK
```

## Const Function

```cpp
class BankAccount {
private:
    double balance;

public:
    // This function promises NOT to change the balance
    double getBalance() const { 
        return balance; 
    }

    // This function DOES change data, so it cannot be const
    void deposit(double amount) {
        balance += amount;
    }
};
```

# Lambda Function

is default a const function that promise won't change the **capture-by-value** `[x]()`. if it wants to modify, it must has keyword `mutable`:

`[x]() mutable {}`

Yes: by default, a lambda’s `operator()` is `const`.

So this:

```cpp
auto f = [x]() {
    return x + 1;
};
```

is roughly like:

```cpp
struct Lambda {
    int x;
    int operator()() const {
        return x + 1;
    }
};
```

## What that means

A default lambda cannot modify its own captured-by-value data.

```cpp
int x = 0;

auto f = [x]() {
    // x++;   // error
};
```

Because the lambda call operator is `const`.

If you want to modify captured-by-value data, use `mutable`:

```cpp
int x = 0;

auto f = [x]() mutable {
    x++;
};
```

## What about class members?

There are 2 common cases.

### 1. Capture `this`

```cpp
class A {
    int value = 0;
public:
    void foo() {
        auto f = [this]() {
            value++;   // OK
        };
        f();
    }
};
```

This is allowed.

Why? Because you are not modifying a captured-by-value field inside the lambda object. You are using the captured pointer `this` to modify the original object.

The lambda is still `const`, but a `const` member function can still use a pointer member to modify the pointee.

Very roughly:

```cpp
struct Lambda {
    A* this_ptr;
    void operator()() const {
        this_ptr->value++;
    }
};
```

The pointer itself is not changed, so `const` is fine.

### 2. Capture `*this` by value

```cpp
class A {
    int value = 0;
public:
    void foo() {
        auto f = [*this]() {
            // value++;   // error
        };
    }
};
```

This captures a copy of the whole object by value.

Now the copied object lives inside the lambda object, and since the lambda call operator is `const`, you cannot modify that copy unless you use `mutable`:

```cpp
auto f = [*this]() mutable {
    value++;   // modifies the copied object inside the lambda
};
```

## Important const case

If the enclosing member function is `const`:

```cpp
class A {
    int value = 0;
public:
    void foo() const {
        auto f = [this]() {
            // value++;   // error
        };
    }
};
```

Then `this` is effectively `const A*`, so you cannot modify `value`.

## Short rule

- lambda default: `operator()` is `const`
- captured-by-value variables: cannot modify unless `mutable`
- captured references: can modify the referred object
- captured `this`: can modify members if the current object is non-const
- captured `*this`: acts like by-value capture, so need `mutable` to modify the copy

If you want, I can show 4 tiny examples side by side:
- `[x]`
- `[&x]`
- `[this]`
- `[*this]`

# Const Iterator

```cpp
// const iterator (can't move iterator)
const vector<int>::iterator iter = v.begin();

// iterator pointing to a constant value
vector<int>::const_iterator iter = v.begin();
```